In [1]:
import pathlib
import sys

_here = pathlib.Path.cwd().resolve()
for _parent in [_here, *_here.parents]:
    if (_parent / "src" / "quant_textbook").exists():
        sys.path.insert(0, str(_parent / "src"))
        break

# 45. Week 31 — Graphical models, mixtures, HMMs, and EM

## 学習目標

- DAGのfactorizationとd-separationを読める
- iid mixtureとHMMの依存構造を区別できる
- forward-backward、Viterbi、Baum–Welchの役割を分けられる
- label switchingとstate occupancy/durationを監査できる

## 前提知識

- B2のMarkov chain
- B7のNS factor changes

In [2]:
import time

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

import quant_textbook as qt

pio.renderers.default = "notebook_connected"
RANDOM_SEED = 20260810
NOTEBOOK_ID = 45


def task_rng(task_id, *coordinates):
    entropy = [
        RANDOM_SEED,
        NOTEBOOK_ID,
        int(task_id),
        *(int(coordinate) for coordinate in coordinates),
    ]
    return np.random.default_rng(np.random.SeedSequence(entropy))

In [3]:
treasury = qt.load_treasury_snapshot()
rates = treasury.frame.copy()
forecast = qt.make_treasury_forecast_dataset(rates)
b5_split = qt.chronological_split(len(forecast.regression_target), gap=1)

maturity_years = np.array([0.25, 2.0, 5.0, 10.0, 30.0])
curve_yields = rates.loc[:, qt.DEFAULT_TENORS].to_numpy(dtype=float)
curve_dates = rates["date"].to_numpy(dtype="datetime64[ns]")
curve_changes_bp = np.diff(curve_yields, axis=0) * 100.0
change_dates = curve_dates[1:]

train_end_date = forecast.prediction_dates[b5_split.train.max()]
validation_end_date = forecast.prediction_dates[b5_split.validation.max()]
test_start_date = forecast.prediction_dates[b5_split.test.min()]
train_mask = curve_dates <= train_end_date
validation_mask = (curve_dates > train_end_date) & (curve_dates <= validation_end_date)
test_mask = curve_dates >= test_start_date

assert treasury.quality.accepted
assert train_end_date < validation_end_date < test_start_date
assert np.all(np.isfinite(curve_yields))
assert np.all(np.diff(curve_dates).astype("timedelta64[D]") > np.timedelta64(0, "D"))

print("source:", treasury.metadata.source_name)
print("snapshot:", treasury.metadata.start_date, "to", treasury.metadata.end_date)
print("curve rows / tenors:", curve_yields.shape)
print("B5 train / validation end:", train_end_date, validation_end_date)
print("locked outer-test start:", test_start_date)
print("snapshot sha256:", treasury.metadata.snapshot_sha256)

source: U.S. Treasury Daily Par Yield Curve Rates
snapshot: 2015-01-02 to 2025-12-31
curve rows / tenors: (2750, 5)
B5 train / validation end: 2021-08-11T00:00:00.000000000 2023-10-19T00:00:00.000000000
locked outer-test start: 2023-10-23T00:00:00.000000000
snapshot sha256: 6ddef9605abbf02c6a4526a51f098135b41da1a437915623af672b1c7bcbd295


## 1. Conditional independence graph

HMMは

$$
p(z_{1:T},y_{1:T})=p(z_1)p(y_1\mid z_1)\prod_{t=2}^T p(z_t\mid z_{t-1})p(y_t\mid z_t)
$$

とfactorizeする。(y_t\perp y_{1:t-1}\mid z_t) はmodel仮定であり、市場の真の生成過程がそうだという観測事実ではない。iid Gaussian mixtureはtransitionを持たず、duration情報を表現しない。

In [4]:
decay = 0.5
factors = qt.extract_nelson_siegel_factors(curve_yields, maturity_years, decay)
factor_changes_bp = np.diff(factors, axis=0) * 100.0
factor_change_dates = curve_dates[1:]
training_rows = factor_change_dates <= train_end_date
audit_rows = factor_change_dates <= validation_end_date

hmm = qt.fit_gaussian_hmm(factor_changes_bp[training_rows], 2)
filtered_probability = qt.hmm_filtered_probabilities(hmm, factor_changes_bp[audit_rows])
smoothed_probability = qt.hmm_smoothed_probabilities(hmm, factor_changes_bp[audit_rows])
diagnostics = qt.hmm_state_diagnostics(hmm, factor_changes_bp[training_rows])
display(
    pd.DataFrame(
        {
            "state": np.arange(2),
            "level_change_mean_bp": hmm.means[:, 0],
            "slope_change_mean_bp": hmm.means[:, 1],
            "curvature_change_mean_bp": hmm.means[:, 2],
            "occupancy": diagnostics.occupancy,
            "mean_viterbi_duration": diagnostics.mean_duration,
        }
    )
)
display(pd.DataFrame(hmm.transition_matrix, index=["from 0", "from 1"], columns=["to 0", "to 1"]))

,state,level_change_mean_bp,slope_change_mean_bp,curvature_change_mean_bp,occupancy,mean_viterbi_duration
0,0,-2.597824,2.931281,-0.581024,0.641475,2.814324
1,1,4.238929,-4.768480,0.758466,0.358525,1.577128


,to 0,to 1
from 0,0.627005,0.372995
from 1,0.623923,0.376077


In [5]:
fig = go.Figure()
dates = factor_change_dates[audit_rows]
fig.add_scatter(x=dates, y=filtered_probability[:, 1], name="filtered P(state 1)", mode="lines")
fig.add_scatter(x=dates, y=smoothed_probability[:, 1], name="smoothed P(state 1)", mode="lines")
fig.update_layout(
    title="HMM state probabilities: online filtering versus retrospective smoothing",
    xaxis_title="Treasury publication date",
    yaxis_title="Conditional probability",
    template="plotly_white",
)
fig.show()

## 2. EM monotonicity and label audit

Baum–WelchはEMであり、観測log likelihoodを減らさない局所更新を行うがglobal optimumは保証しない。実装はlevel-factor change meanでlabelをcanonicalizeする。別initializationでlabelが反転してもlikelihoodは同じになり得る。

In [6]:
assert np.all(np.diff(hmm.log_likelihood_trace) >= -1e-6)
assert np.all(np.diff(hmm.means[:, 0]) >= 0.0)
em_table = pd.DataFrame(
    {
        "iteration": np.arange(1, len(hmm.log_likelihood_trace) + 1),
        "log_likelihood": hmm.log_likelihood_trace,
    }
)
display(em_table.tail())
print("converged:", hmm.converged)
print("labels are observed truth:", False)

,iteration,log_likelihood
8,9,-16291.324332
9,10,-16291.149936
10,11,-16291.091068
11,12,-16291.069740
12,13,-16291.060774


converged: True
labels are observed truth: False


## 3. 失敗モード

- smoothed probabilityをforecast-origin stateへ使う
- EM convergenceをglobal optimumやBayesian posteriorと呼ぶ
- state 0/1の番号に経済的意味を固定する
- occupancyが極小のstateを説明せず残す
- state数をouter test likelihoodで選ぶ

## 4. 段階別演習

### 基礎

1. forward recursionをlog-sum-expで書け。
2. filtered/smoothed/Viterbiの目的を比較せよ。

### 標準

3. 2/3/4 stateをtraining fit、validation log scoreで比較せよ。
4. state durationとgeometric distributionの関係を導出せよ。

### 研究

5. switching VARまたはswitching Kalman modelの識別条件を書け。

## 5. Exit Criteria

- [ ] HMM factorizationを書ける
- [ ] iid mixtureとMarkov mixtureを区別した
- [ ] EM log likelihoodの単調性を検査した
- [ ] label canonicalizationとstate数感応度を計画した
- [ ] filtered stateだけがforecast originで利用可能と説明した

## 6. 出典


- [Rabiner (1989), A Tutorial on Hidden Markov Models](https://www.cs.cmu.edu/~durand/03-711/Readings/Rabiner89.pdf)
- [Vehtari et al., Rank-normalization, folding, and localization](https://arxiv.org/abs/1903.08008)
- [Stan Reference Manual — MCMC Sampling](https://mc-stan.org/docs/reference-manual/mcmc.html)